In [1]:
### 0. IMPORTAÇÃO DE BIBLIOTECAS ###
import pandas as pd
import numpy as np
import unicodedata
import re
import sys

print(f'Executável do Python: {sys.executable}')

Executável do Python: C:\Users\luish\anaconda3\envs\bs_concorrente\python.exe


In [2]:
### 1. CARGA DOS DADOS BRUTOS ###
df = pd.read_excel(r"C:\Users\luish\Downloads\Desenvolvimento TCC\TCC\01 - Dados Brutos\Secundarios\Base de dados\base_concorrente_v1.xlsx")
print(f'Dataset Concorrentes carregado: {df.shape[0]} linhas e {df.shape[1]} colunas.\n')

Dataset Concorrentes carregado: 1098 linhas e 39 colunas.



In [3]:
### 2. EXPLORAÇÃO E DIAGNÓSTICO INICIAL ###
print("="*50)
print("--- DIAGNÓSTICO INICIAL DOS DADOS (CONCORRENTES) ---")
print("="*50)

print("\n[INFO] Estrutura do DataFrame:")
df.info()

print("\n[INFO] Primeiras 3 linhas:")
print(df.head(3))

print("\n[INFO] Percentual de Valores Nulos por Coluna:")
nulos_pct = (df.isnull().sum() / len(df)) * 100
print(nulos_pct[nulos_pct > 0].round(2).to_string() if not nulos_pct[nulos_pct > 0].empty else "Nenhum valor nulo encontrado.")

print("\n[INFO] Total de Linhas Duplicadas no Dataset Bruto:", df.duplicated().sum())
print("="*50 + "\n")

--- DIAGNÓSTICO INICIAL DOS DADOS (CONCORRENTES) ---

[INFO] Estrutura do DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 1098 entries, 0 to 1097
Data columns (total 39 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   CNPJ                                  1098 non-null   int64  
 1   Raiz CNPJ                             1098 non-null   int64  
 2   Numero da Filial                      1098 non-null   int64  
 3   Razao Social                          1098 non-null   str    
 4   Matriz Filial                         1098 non-null   str    
 5   Codigo Natureza Juridica              1098 non-null   int64  
 6   Descricao Natureza Juridica           1098 non-null   str    
 7   Nome Fantasia                         581 non-null    str    
 8   Situacao Cadastral                    1098 non-null   str    
 9   Data Situacao Cadastral               1098 non-null   str    
 10  Motivo Sit

In [4]:
### 3. PADRONIZAÇÃO E NORMALIZAÇÃO DE NOMES DAS COLUNAS ###
def normalizar_nome_coluna(col):
    if not isinstance(col, str):
        return col
    nfkd = unicodedata.normalize('NFKD', col)
    texto_sem_acento = u"".join([c for c in nfkd if not unicodedata.combining(c)])
    texto_limpo = re.sub(r'[^a-zA-Z0-9]+', '_', texto_sem_acento).strip('_').lower()
    return texto_limpo

df.columns = [normalizar_nome_coluna(col) for col in df.columns]
print("[OK] Nomes de colunas padronizados para snake_case e sem acentos.")

[OK] Nomes de colunas padronizados para snake_case e sem acentos.


In [5]:
### 4. AJUSTE DE TIPAGEM (DATA TYPES) ###
df_limpo = df.copy()

if 'cnpj' in df_limpo.columns:
    df_limpo['cnpj'] = df_limpo['cnpj'].astype(str).str.replace(r'\D', '', regex=True).str.zfill(14)
if 'cep' in df_limpo.columns:
    df_limpo['cep'] = df_limpo['cep'].astype(str).str.replace(r'\D', '', regex=True).str.zfill(8)

if 'data_de_abertura' in df_limpo.columns:
    df_limpo['data_de_abertura'] = pd.to_datetime(df_limpo['data_de_abertura'], errors='coerce')
if 'data_situacao_cadastral' in df_limpo.columns:
    df_limpo['data_situacao_cadastral'] = pd.to_datetime(df_limpo['data_situacao_cadastral'], errors='coerce')

# --- LOGS DE VALIDAÇÃO DO BLOCO 4 ---
print("\n[OK] Tipagem e formatadores ajustados.")
cols_val_tipagem = [c for c in ['cnpj', 'cep', 'data_de_abertura'] if c in df_limpo.columns]
print(df_limpo[cols_val_tipagem].dtypes)
print("Amostra formatada:")
print(df_limpo[cols_val_tipagem].head(3))


[OK] Tipagem e formatadores ajustados.
cnpj                           str
cep                            str
data_de_abertura    datetime64[us]
dtype: object
Amostra formatada:
             cnpj       cep data_de_abertura
0  68924564000154  68515000       2026-08-28
1  55314916000108  68515000       2024-05-28
2  36140744000159  68515000       2020-01-27


In [6]:
### 5. SANEAMENTO E FILTRAGEM DE EMPRESAS ATIVAS ###
# Filtrar empresas ATIVAS
if 'situacao_cadastral' in df_limpo.columns:
    df_limpo['situacao_cadastral'] = df_limpo['situacao_cadastral'].astype(str).str.strip().str.upper()
    df_limpo = df_limpo[df_limpo['situacao_cadastral'] == 'ATIVA']
    print(f"\n[OK] Linhas ativas mantidas: {len(df_limpo)}")

# Remover Duplicações pelo CNPJ
if 'cnpj' in df_limpo.columns:
    df_limpo = df_limpo.drop_duplicates(subset=['cnpj'], keep='first')
    print(f"[OK] Linhas após remover CNPJs duplicados: {len(df_limpo)}")


[OK] Linhas ativas mantidas: 1098
[OK] Linhas após remover CNPJs duplicados: 1098


In [7]:
### 6. REMOÇÃO DE COLUNAS IRRELEVANTES (PÓS-DIAGNÓSTICO E PÓS-FILTRAGEM) ###
colunas_para_remover = [
    'situacao_especial', 'ente_federativo_responsavel', 'raiz_cnpj', 
    'numero_da_filial', 'matriz_filial', 'motivo_situacao_cadastral', 
    'e_mail', 'email', 'data_da_consulta', 'socios', 
    'data_de_opcao_mei', 'data_exclusao_mei', 'data_de_opcao_simples', 
    'data_exclusao_simples', 'situacao_cadastral'
]

df_limpo = df_limpo.drop(columns=[col for col in colunas_para_remover if col in df_limpo.columns], errors='ignore')
print(f"[OK] Colunas irrelevantes removidas. Restantes: {df_limpo.shape[1]} colunas.")

[OK] Colunas irrelevantes removidas. Restantes: 25 colunas.


In [8]:
### 7. TRATAMENTO DE VALORES AUSENTES (MISSING DATA) ###
if 'razao_social' in df_limpo.columns and 'nome_fantasia' in df_limpo.columns:
    nulos_fantasia = df_limpo['nome_fantasia'].isna().sum()
    df_limpo['nome_fantasia'] = df_limpo['nome_fantasia'].fillna(df_limpo['razao_social'])
    print(f"\n[OK] 'nome_fantasia': {nulos_fantasia} registros nulos preenchidos com a 'razao_social'.")

valores_padrao = {
    'complemento': 'NÃO INFORMADO',
    'bairro': 'NÃO INFORMADO'
}

nulos_antes = {col: df_limpo[col].isna().sum() for col in valores_padrao.keys() if col in df_limpo.columns}
df_limpo = df_limpo.fillna(value=valores_padrao)

for col, total in nulos_antes.items():
    print(f"[OK] '{col}': {total} registros ausentes substituídos por 'NÃO INFORMADO'.")


[OK] 'nome_fantasia': 517 registros nulos preenchidos com a 'razao_social'.
[OK] 'complemento': 295 registros ausentes substituídos por 'NÃO INFORMADO'.
[OK] 'bairro': 0 registros ausentes substituídos por 'NÃO INFORMADO'.


In [9]:
### 8. TRATAMENTO AVANÇADO DE COMPLEMENTO E LOGRADOURO ###
def limpar_texto_endereco(texto):
    if pd.isna(texto) or str(texto).strip() == '' or str(texto).upper() == 'NAN':
        return ''
    
    texto = str(texto).upper().strip()
    
    # Padronizar separadores e adicionar espaços entre palavras/números colados
    texto = re.sub(r';', ' ', texto)
    texto = re.sub(r'(QUADRA|QD|Q|LOTE|LT|L)(\d+)', r'\1 \2', texto)
    
    # Padronizar abreviações comuns
    texto = re.sub(r'\bQD\.?\b', 'QUADRA', texto)
    texto = re.sub(r'\bLT\.?\b', 'LOTE', texto)
    texto = re.sub(r'\bKM\.?\b', 'KM', texto)
    
    # Remover duplicidade de prefixos de logradouro
    texto = re.sub(r'\b(RUA)\s+\1\b', r'\1', texto)
    texto = re.sub(r'\b(AVENIDA)\s+\1\b', r'\1', texto)
    
    # Eliminar múltiplos espaços
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

# Padronizar colunas de texto gerais
colunas_texto = ['razao_social', 'nome_fantasia', 'bairro', 'municipio']
for col in colunas_texto:
    if col in df_limpo.columns:
        df_limpo[col] = df_limpo[col].astype(str).str.strip().str.upper()

# Aplicar Regex avançada no Logradouro e Complemento
if 'logradouro' in df_limpo.columns:
    df_limpo['logradouro'] = df_limpo['logradouro'].apply(limpar_texto_endereco)

if 'complemento' in df_limpo.columns:
    df_limpo['complemento'] = df_limpo['complemento'].apply(limpar_texto_endereco)
    df_limpo['complemento'] = df_limpo['complemento'].replace('', 'NÃO INFORMADO')

# --- LOGS DE VALIDAÇÃO DO BLOCO 8 ---
print("\n[OK] Limpeza de Regex aplicada em 'logradouro' e 'complemento'.")
cols_end = [c for c in ['logradouro', 'numero', 'complemento'] if c in df_limpo.columns]
print("Amostra de Logradouro e Complemento saneados:")
print(df_limpo[cols_end].head(3))


[OK] Limpeza de Regex aplicada em 'logradouro' e 'complemento'.
Amostra de Logradouro e Complemento saneados:
                  logradouro numero  \
0  RODOVIA DR. FAISAL SALMEN     SN   
1                      RUA 5    239   
2                  RUA CEARA     38   

                                    complemento  
0  QUADRA 02 LOTE 01 SALA 03 GALERIA NEW CENTER  
1                             QUADRA 36 LOTE 18  
2                             QUADRA 79 LOTE 40  


In [10]:
### 9. ESTRUTURAÇÃO DE ENDEREÇO QGIS E CLASSIFICAÇÃO POR TIPO DE CONCORRÊNCIA ###
if 'municipio' not in df_limpo.columns:
    df_limpo['municipio'] = 'PARAUAPEBAS'
if 'uf' not in df_limpo.columns:
    df_limpo['uf'] = 'PA'

# Concatenação limpa sem incluir 'NÃO INFORMADO'
def montar_endereco_qgis(row):
    comp = f" - {row['complemento']}" if row['complemento'] != 'NÃO INFORMADO' else ""
    return (
        f"{row['logradouro']}, {row['numero']}{comp} - "
        f"{row['bairro']}, {row['municipio']} - {row['uf']}, "
        f"{row['cep']}, Brasil"
    )

df_limpo['endereco_completo'] = df_limpo.apply(montar_endereco_qgis, axis=1)

print("\n[OK] Coluna 'endereco_completo' gerada para geocodificação QGIS!")
print(f"Exemplo de saída: {df_limpo['endereco_completo'].iloc[0]}")

# --- MAPEAMENTO E FILTRAGEM ATUALIZADA DE CNAES ---
cnaes_direto = ["8650099", "8630504", "8650001", "8630503", "8630501"]
cnaes_indireto = ["9602502", "8690901", "8650004"]

def classificar_concorrencia(row):
    principal = str(row.get('codigo_da_atividade_principal', ''))
    secundarias = str(row.get('codigos_das_atividades_secundarias', ''))
    
    # Remove qualquer caractere não numérico para garantir a busca dos CNAEs
    texto_cnaes = re.sub(r'\D', ' ', principal + ' ' + secundarias)
    
    # Checa primeiro os CNAEs Diretos
    if any(cnae in texto_cnaes for cnae in cnaes_direto):
        return 'Direto'
    # Em seguida os CNAEs Indiretos
    elif any(cnae in texto_cnaes for cnae in cnaes_indireto):
        return 'Indireto'
    return 'Outros'

# Aplica a função de classificação e gera a nova coluna
df_limpo['tipo_concorrencia'] = df_limpo.apply(classificar_concorrencia, axis=1)

# Separar os dataframes por tipo
df_concorrente_direto = df_limpo[df_limpo['tipo_concorrencia'] == 'Direto'].copy()
df_concorrente_indireto = df_limpo[df_limpo['tipo_concorrencia'] == 'Indireto'].copy()

# Logs dos totais classificados
print("\n[OK] Classificação por Tipo de Concorrência concluída:")
print(f"   -> Diretos encontrados: {len(df_concorrente_direto)}")
print(f"   -> Indiretos encontrados: {len(df_concorrente_indireto)}")


[OK] Coluna 'endereco_completo' gerada para geocodificação QGIS!
Exemplo de saída: RODOVIA DR. FAISAL SALMEN, SN - QUADRA 02 LOTE 01 SALA 03 GALERIA NEW CENTER - CIDADE JARDIM, PARAUAPEBAS - PA, 68515000, Brasil

[OK] Classificação por Tipo de Concorrência concluída:
   -> Diretos encontrados: 463
   -> Indiretos encontrados: 627


In [11]:
### 10. EXPORTAÇÃO DOS RESULTADOS ###
caminho_direto = r"C:\Users\luish\concorrente_direto.xlsx"
caminho_indireto = r"C:\Users\luish\concorrente_indireto.xlsx"

df_concorrente_direto.to_excel(caminho_direto, index=False)
df_concorrente_indireto.to_excel(caminho_indireto, index=False)

print("\n" + "="*50)
print("[OK] PIPELINE DE CONCORRENTES EXECUTADA COM SUCESSO!")
print("="*50)
print(f"-> Concorrentes Diretos salvos: {len(df_concorrente_direto)} linhas em {caminho_direto}")
print(f"-> Concorrentes Indiretos salvos: {len(df_concorrente_indireto)} linhas em {caminho_indireto}")


[OK] PIPELINE DE CONCORRENTES EXECUTADA COM SUCESSO!
-> Concorrentes Diretos salvos: 463 linhas em C:\Users\luish\concorrente_direto.xlsx
-> Concorrentes Indiretos salvos: 627 linhas em C:\Users\luish\concorrente_indireto.xlsx
